In [ ]:
from typing import Annotated # Importing Annotated for message handling
from dotenv import load_dotenv 
load_dotenv() # Load environment variables from .env file

from typing import TypedDict # Import TypedDict for state structure
from langchain.chat_models import init_chat_model # Import chat model initializer

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages # add_messages automatically stores conversation history

In [ ]:
# Initialize Gemini model
llm = init_chat_model("google_genai:gemini-2.5-flash")

# Define graph state structure
class State(TypedDict):
    messages: Annotated[list, add_messages]    # Stores all chat messages

def chatbot(state: State) -> State:
    # Send messages to LLM and get response
    return {"messages": [llm.invoke(state["messages"])]}    


builder = StateGraph(State)
builder.add_node("chatbot_node", chatbot)

builder.add_edge(START, "chatbot_node")
builder.add_edge("chatbot_node", END)

graph = builder.compile()

In [ ]:
# Create user message
message = {"role": "user", "content": "Who walked on the moon for the first time? Print only the name"}
# message = {"role": "user", "content": "What is the latest price of MSFT stock?"}
response = graph.invoke({"messages":[message]})

response["messages"]

[HumanMessage(content='Who walked on the moon for the first time? Print only the name', additional_kwargs={}, response_metadata={}, id='1c79f3b0-337a-4cf7-8f4e-f3232cb71677'),
 AIMessage(content='Neil Armstrong', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--d6bb6724-3b24-4815-aae1-2ffaed8b0089-0', usage_metadata={'input_tokens': 14, 'output_tokens': 3, 'total_tokens': 17, 'input_token_details': {'cache_read': 0}})]

In [ ]:
state = None
while True:
    in_message = input("You: ")

    # Stop chatbot if user types quit or exit
    if in_message.lower() in {"quit","exit"}:
        break

    # First conversation message
    if state is None:
        state: State = {
            "messages": [{"role": "user", "content": in_message}]
        }
    else:
        # Add new user message to existing conversation
        state["messages"].append({"role": "user", "content": in_message})

    state = graph.invoke(state)

    # Print chatbot response
    print("Bot:", state["messages"][-1].content)